# 01 - 基于物理方法的CN增强星候选体筛选



**方法概述：** 利用LAMOST光谱的CN/CH分子带特征，通过面积筛选和指数筛选两步物理方法，在不依赖已知样本训练的情况下，直接从光谱形态识别CN增强星候选体。



**核心思路：**

1. **分子带面积筛选**：计算CN3839、CN4142、CH4300三个分子带相对连续谱的吸收面积，利用聚类内z-score筛选异常强吸收体

2. **LAMOST指数筛选**：计算传统LAMOST CN/CH指数，二次验证候选体

3. **分子带掩盖聚类**：先掩盖CN/CH分子带波段 → PCA降维 → KMeans聚类，避免分子带特征主导聚类结果。所有后续z-score和排名筛选均在masked_cluster内进行，确保比较的是相似连续谱形态的恒星



**与ML方法的互补性：** 物理方法不依赖训练标签，可作为ML候选体的独立验证手段。

In [ ]:
# 共享数据加载与基础库导入

import sys, os

from pathlib import Path



# 智能定位项目根目录：向上查找直到找到 stars.csv 或 spectra.py

_PROJECT_ROOT = Path(os.getcwd())

for _ in range(5):

    if (_PROJECT_ROOT / "stars.csv").exists() or (_PROJECT_ROOT / "spectra.py").exists():

        break

    _PROJECT_ROOT = _PROJECT_ROOT.parent

if str(_PROJECT_ROOT) not in sys.path:

    sys.path.insert(0, str(_PROJECT_ROOT))



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import matplotlib

matplotlib.rcParams.update({'font.size': 10})

plt.rcParams.update({'axes.labelsize': 'large'})

import warnings

warnings.filterwarnings('ignore')



# 加载共享数据

from PhaseSummary.shared.data_loader import ensure_cache, BAND_DEFS, MOLECULAR_BAND_RANGES



data = ensure_cache()

X_clean = data['X_clean']

stars_clean = data['stars_clean']

feature_df = data['feature_df']

common_wave = data['common_wave']



print(f"数据加载完成:")

print(f"  光谱矩阵: {X_clean.shape}")

print(f"  恒星数量: {len(stars_clean)}")

print(f"  已知CN星: {(stars_clean['label']==1).sum()}")

print(f"  波长范围: {common_wave[0]:.0f}-{common_wave[-1]:.0f} Å")



## 1. 分子带面积计算



对每条光谱，用连续谱归一化后计算分子带相对于两侧连续谱的吸收面积。CN增强星在这些分子带有显著更强的吸收。

In [ ]:
# 分子带面积计算

from scipy import signal



band_cn = {

    'CN3839': (3830, 3883),

    'CN4142': (4120, 4216),

    'CH4300': (4285, 4315),

}



def _safe_smooth(y, win=9, poly=2):

    y = np.asarray(y, dtype=float)

    w = int(win)

    if w % 2 == 0:

        w += 1

    if len(y) < w:

        return y

    return signal.savgol_filter(y, w, poly)



def compute_band_area(flux, wave, band_range):

    """计算分子带相对于连续谱的吸收面积（正值表示吸收）。"""

    l1, l2 = band_range

    mask_band = (wave >= l1) & (wave <= l2)

    if mask_band.sum() < 5:

        return np.nan



    # 连续谱：带外两侧线性插值

    mask_left = wave < l1

    mask_right = wave > l2

    if mask_left.sum() < 3 or mask_right.sum() < 3:

        return np.nan



    cont_left = np.median(flux[mask_left][-5:]) if mask_left.sum() >= 5 else np.median(flux[mask_left])

    cont_right = np.median(flux[mask_right][:5]) if mask_right.sum() >= 5 else np.median(flux[mask_right])



    wave_band = wave[mask_band]

    continuum = np.interp(wave_band, [l1, l2], [cont_left, cont_right])

    flux_band = _safe_smooth(flux[mask_band])

    flux_band = flux_band[:len(continuum)]



    area = np.trapz(np.maximum(continuum - flux_band, 0), wave_band[:len(flux_band)])

    return area



# 计算所有光谱的三个分子带面积

n_stars = len(X_clean)

print(f"计算 {n_stars} 条光谱的分子带面积...")



area_cn3839 = np.array([compute_band_area(X_clean[i], common_wave, band_cn['CN3839']) for i in range(n_stars)])

area_cn4142 = np.array([compute_band_area(X_clean[i], common_wave, band_cn['CN4142']) for i in range(n_stars)])

area_ch4300 = np.array([compute_band_area(X_clean[i], common_wave, band_cn['CH4300']) for i in range(n_stars)])



# 填充NaN

for arr in [area_cn3839, area_cn4142, area_ch4300]:

    arr[np.isnan(arr)] = np.nanmedian(arr)



print(f"面积范围: CN3839 [{area_cn3839.min():.4f}, {area_cn3839.max():.4f}]")

print(f"           CN4142 [{area_cn4142.min():.4f}, {area_cn4142.max():.4f}]")

print(f"           CH4300 [{area_ch4300.min():.4f}, {area_ch4300.max():.4f}]")



# 保存到DataFrame

area_df = stars_clean[['ra', 'dec', 'teff', 'logg', 'feh', 'label', 'uid', 'masked_cluster_id']].copy()

area_df['area_CN3839'] = area_cn3839

area_df['area_CN4142'] = area_cn4142

area_df['area_CH4300'] = area_ch4300



## 2. 聚类内z-score筛选



在每个分子带聚类（masked_cluster）内部计算z-score，筛选异常强吸收的恒星作为候选体。

In [ ]:
# 聚类内z-score筛选

from scipy.stats import zscore



def cluster_zscore_filter(df, value_col, cluster_col='masked_cluster_id', threshold=2.5):

    """在每个聚类内计算z-score，返回超过阈值的候选体标记。"""

    z_scores = np.zeros(len(df))

    for cid in df[cluster_col].unique():

        mask = df[cluster_col] == cid

        vals = df.loc[mask, value_col].values

        if len(vals) < 10:

            continue

        z_scores[mask] = np.abs(zscore(vals, nan_policy='omit'))

    return z_scores >= threshold



# CN3839 + CN4142 联合筛选

z_cn3839 = cluster_zscore_filter(area_df, 'area_CN3839', threshold=2.5)

z_cn4142 = cluster_zscore_filter(area_df, 'area_CN4142', threshold=2.5)

z_ch4300 = cluster_zscore_filter(area_df, 'area_CH4300', threshold=2.0)



# CN增强候选：CN3839或CN4142显著吸收，且CH4300不过度偏离

candidate_mask = (z_cn3839 | z_cn4142) & (~z_ch4300)

area_df['area_candidate'] = candidate_mask.astype(int)



n_cand = candidate_mask.sum()

n_known = (area_df['label'] == 1).sum()

n_known_in_cand = ((area_df['label'] == 1) & candidate_mask).sum()

print(f"面积法候选体: {n_cand} ({n_cand/len(area_df)*100:.2f}%)")

print(f"已知CN星总数: {n_known}, 被面积法召回: {n_known_in_cand} ({n_known_in_cand/n_known*100:.1f}%)")



## 3. 可视化：分子带面积分布

In [ ]:
# 可视化：CN3839 vs CN4142 面积散点图

fig, axes = plt.subplots(1, 3, figsize=(18, 5))



vis = area_df.copy()

m_unl = vis['label'] == -1

m_kn = vis['label'] == 1

m_cand = vis['area_candidate'] == 1



# CN3839 vs CN4142

axes[0].scatter(vis.loc[m_unl, 'area_CN3839'], vis.loc[m_unl, 'area_CN4142'],

                s=8, alpha=0.15, c='#95a5a6', edgecolors='none', label='Unlabeled')

axes[0].scatter(vis.loc[m_kn, 'area_CN3839'], vis.loc[m_kn, 'area_CN4142'],

                s=60, marker='*', c='#e74c3c', edgecolors='black', linewidth=0.5,

                label=f'Known CN (n={m_kn.sum()})', zorder=3)

axes[0].scatter(vis.loc[m_cand, 'area_CN3839'], vis.loc[m_cand, 'area_CN4142'],

                s=20, facecolors='none', edgecolors='#2980b9', linewidth=0.8,

                label=f'Candidates (n={m_cand.sum()})', zorder=2)

axes[0].set_xlabel('CN3839 Band Area')

axes[0].set_ylabel('CN4142 Band Area')

axes[0].set_title('CN3839 vs CN4142')

axes[0].legend(fontsize=7)

axes[0].grid(alpha=0.2)



# CN3839 vs CH4300

axes[1].scatter(vis.loc[m_unl, 'area_CN3839'], vis.loc[m_unl, 'area_CH4300'],

                s=8, alpha=0.15, c='#95a5a6', edgecolors='none')

axes[1].scatter(vis.loc[m_kn, 'area_CN3839'], vis.loc[m_kn, 'area_CH4300'],

                s=60, marker='*', c='#e74c3c', edgecolors='black', linewidth=0.5, zorder=3)

axes[1].scatter(vis.loc[m_cand, 'area_CN3839'], vis.loc[m_cand, 'area_CH4300'],

                s=20, facecolors='none', edgecolors='#2980b9', linewidth=0.8, zorder=2)

axes[1].set_xlabel('CN3839 Band Area')

axes[1].set_ylabel('CH4300 Band Area')

axes[1].set_title('CN3839 vs CH4300')

axes[1].grid(alpha=0.2)



# Teff-logg分布

axes[2].scatter(vis.loc[m_unl, 'teff'], vis.loc[m_unl, 'logg'],

                s=5, alpha=0.12, c='#95a5a6', edgecolors='none', label='Unlabeled')

axes[2].scatter(vis.loc[m_kn, 'teff'], vis.loc[m_kn, 'logg'],

                s=60, marker='*', c='#e74c3c', edgecolors='black', linewidth=0.5,

                label='Known CN', zorder=3)

axes[2].scatter(vis.loc[m_cand, 'teff'], vis.loc[m_cand, 'logg'],

                s=18, facecolors='none', edgecolors='#2980b9', linewidth=0.8,

                label='Candidates', zorder=2)

axes[2].set_xlabel('Teff (K)')

axes[2].set_ylabel('log g')

axes[2].set_title('Teff-logg Distribution')

axes[2].invert_xaxis()

axes[2].invert_yaxis()

axes[2].legend(fontsize=7)

axes[2].grid(alpha=0.2)



plt.suptitle('Physics-based CN Candidate Screening (Area Method)', fontsize=14, y=1.01)

plt.tight_layout()

plt.savefig(_PROJECT_ROOT / 'PhaseSummary/01_T_physics/area_screening.png', dpi=150, bbox_inches='tight')

plt.show()

print("面积法筛选可视化已保存")



## 4. CN/CH指数二级筛选



在面积法初筛基础上，计算传统LAMOST指数（CN3839, CN4142, CH4300），进行聚类内排名筛选，提高候选体可靠性。

In [ ]:
# CN/CH 指数计算与二级筛选

from PhaseSummary.shared.data_loader import compute_cn_indices



cn3839, cn4142, ch4300 = compute_cn_indices(X_clean, common_wave)



area_df['CN3839_idx'] = cn3839

area_df['CN4142_idx'] = cn4142

area_df['CH4300_idx'] = ch4300



# 在候选体内按聚类进行排名筛选

def cluster_rank_filter(df, value_col, cluster_col='masked_cluster_id', top_frac=0.05):

    """在每个聚类内保留top-fraction的最高值样本。"""

    keep = np.zeros(len(df), dtype=bool)

    for cid in df[cluster_col].unique():

        mask = (df[cluster_col] == cid) & (df['area_candidate'] == 1)

        if mask.sum() < 3:

            continue

        n_keep = max(1, int(mask.sum() * top_frac))

        cluster_idx = df.index[mask]

        vals = df.loc[mask, value_col].values

        top_local = cluster_idx[np.argsort(vals)[-n_keep:]]

        keep[top_local] = True

    return keep



# CN3839 + CN4142联合排名

keep_cn3839 = cluster_rank_filter(area_df, 'CN3839_idx', top_frac=0.03)

keep_cn4142 = cluster_rank_filter(area_df, 'CN4142_idx', top_frac=0.03)

final_mask = keep_cn3839 | keep_cn4142



area_df['final_candidate'] = final_mask.astype(int)



n_final = final_mask.sum()

n_known_final = ((area_df['label'] == 1) & final_mask).sum()

print(f"最终候选体: {n_final} ({n_final/len(area_df)*100:.2f}%)")

print(f"已知CN星召回: {n_known_final}/{n_known} ({n_known_final/n_known*100:.1f}%)")

print(f"筛选率: {n_cand} → {n_final} (缩减 {n_cand-n_final})")



## 5. 候选体光谱可视化

In [ ]:
# 候选体光谱可视化

cand_indices = area_df.index[area_df['final_candidate'] == 1]

n_show = min(16, len(cand_indices))

np.random.seed(42)

show_idx = np.random.choice(cand_indices, n_show, replace=False)



fig, axes = plt.subplots(4, 4, figsize=(16, 12))

axes = axes.flatten()



# 预计算每个簇的平均光谱

cluster_means = {}

for cid in area_df['masked_cluster_id'].unique():

    cmask = area_df['masked_cluster_id'] == cid

    cluster_means[cid] = np.nanmedian(X_clean[cmask.values], axis=0)



for i, idx in enumerate(show_idx):

    ax = axes[i]

    flux = X_clean[idx]

    cid = area_df.loc[idx, 'masked_cluster_id']



    # 簇内平均光谱（灰色虚线，用于对比CN增峰）

    if cid in cluster_means:

        ax.plot(common_wave, cluster_means[cid], color='darkorange',

                linewidth=1.0, linestyle='--', alpha=0.85, label='Cluster mean')



    ax.plot(common_wave, flux, color='navy', linewidth=0.8, label='Candidate')



    # 分子带标记

    for l1, l2, c, name in [(3830, 3883, 'blue', 'CN3839'),

                               (4120, 4216, 'green', 'CN4142'),

                               (4285, 4315, 'red', 'CH4300')]:

        ax.axvspan(l1, l2, alpha=0.12, color=c, zorder=0)



    teff = area_df.loc[idx, 'teff']

    cn3839_v = area_df.loc[idx, 'CN3839_idx']

    ax.set_title(f'#{i+1} | Teff={teff:.0f}K | CN3839={cn3839_v:.3f}', fontsize=8)

    ax.set_xlim(3800, 4500)

    ax.tick_params(labelsize=7)

    ax.grid(alpha=0.2)



    if i == 0:

        ax.legend(fontsize=6, loc='upper right')



for j in range(n_show, len(axes)):

    axes[j].axis('off')



fig.suptitle('Physics-based CN Candidate Spectra (Final Selection)', fontsize=14, y=1.01)

plt.tight_layout()

plt.savefig(_PROJECT_ROOT / 'PhaseSummary/01_T_physics/candidate_spectra.png', dpi=150, bbox_inches='tight')

plt.show()

print("候选体光谱可视化已保存")



## 6. 导出候选体



导出物理方法筛选的候选体列表，供ML方法交叉验证使用。

In [ ]:
# 导出候选体

export_cols = ['ra', 'dec', 'teff', 'logg', 'feh', 'uid', 'label',

              'area_CN3839', 'area_CN4142', 'area_CH4300',

              'CN3839_idx', 'CN4142_idx', 'CH4300_idx',

              'masked_cluster_id', 'final_candidate']



export_df = area_df[export_cols].copy()

export_df = export_df.sort_values('area_CN3839', ascending=False)



# 保存完整表

outpath = str(_PROJECT_ROOT / 'PhaseSummary/01_T_physics/T_physics_candidates.csv')

export_df.to_csv(outpath, index=False)



# 仅候选体

cand_export = export_df[export_df['final_candidate'] == 1].copy()

cand_outpath = str(_PROJECT_ROOT / 'PhaseSummary/01_T_physics/T_physics_candidates_only.csv')

cand_export.to_csv(cand_outpath, index=False)



print(f"完整表已导出: {outpath} ({len(export_df)} 行)")

print(f"候选体表已导出: {cand_outpath} ({len(cand_export)} 行)")

print(f"\n候选体统计:")

print(f"  总数: {len(cand_export)}")

print(f"  其中已知CN星: {(cand_export['label']==1).sum()}")

print(f"  新候选体: {(cand_export['label']!=1).sum()}")

print(f"  Teff范围: {cand_export['teff'].min():.0f} - {cand_export['teff'].max():.0f} K")

print(f"  logg范围: {cand_export['logg'].min():.2f} - {cand_export['logg'].max():.2f}")



## 7. 结论



**物理方法（T_physics）总结：**



1. **方法特点**：不依赖训练标签，直接从光谱CN/CH分子带形态出发识别候选体

2. **面积法初筛**：利用聚类内z-score > 2.5筛选异常CN吸收体，排除CH4300异常的碳星污染

3. **指数法复筛**：在初筛基础上用传统CN指数进行聚类内top-3%排名筛选

4. **优势**：可与ML方法形成互补验证，物理上可解释

5. **局限性**：依赖连续谱归一化质量，对低SNR光谱敏感



导出的候选体列表可用于与ML方法（Notebook 03）进行交叉验证。